# 05. LSTM Training

Training LSTM neural networks using Keras. We use a 30-day lookback window.

In [ ]:
import pandas as pd
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler
import joblib
import os

df = pd.read_csv("../backend/data/processed/engineered_features.csv")
df["date"] = pd.to_datetime(df["date"])

def create_sequences(data, seq_length=30):
    xs, ys = [], []
    for i in range(len(data)-seq_length):
        xs.append(data[i:(i+seq_length)])
        ys.append(data[i+seq_length])
    return np.array(xs), np.array(ys)

def train_lstm(commodity):
    print(f"Training LSTM for {commodity}...")
    data = df[df["commodity"] == commodity].sort_values("date")["price"].values.reshape(-1, 1)
    
    scaler = MinMaxScaler()
    data_scaled = scaler.fit_transform(data)
    joblib.dump(scaler, f"../backend/models/saved/scaler_{commodity}.pkl")
    
    X, y = create_sequences(data_scaled)
    split = int(len(X) * 0.8)
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]
    
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(30, 1)),
        Dropout(0.2),
        LSTM(32),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    
    model.compile(optimizer='adam', loss='huber')
    es = EarlyStopping(patience=10, restore_best_weights=True)
    
    model.fit(X_train, y_train, epochs=20, batch_size=32, validation_data=(X_test, y_test), callbacks=[es], verbose=0)
    
    model.save(f"../backend/models/saved/lstm_{commodity}.h5")
    print(f"LSTM saved for {commodity}.")

for c in ["onion", "potato", "tomato"]:
    train_lstm(c)